# IMT 573 - Problem Set 8 - Prediction

### Instructions

Before beginning this assignment, please ensure you have access to a working instance of Jupyter Notebooks with Python 3.

1. First, replace the “YOUR NAME HERE” text in the next cell with your own full name. Any collaborators must also be listed in this cell.

2. Be sure to include well-documented (e.g. commented) code cells, figures, and clearly written text  explanations as necessary. Any figures should be clearly labeled and appropriately referenced within the text. Be sure that each visualization adds value to your written explanation; avoid redundancy – you do no need four different visualizations of the same pattern.

3. Collaboration on problem sets and labs is fun, useful, and encouraged. However, each student must turn in an individual write-up in their own words as well as code/work that is their own. Regardless of whether you work with others, what you turn in must be your own work; this includes code and interpretation of results. The names of all collaborators must be listed on each assignment. Do not copy-and-paste from other students’ responses or code - your code should never be on any other student's screen or machine.

4. All materials and resources that you use (with the exception of lecture slides) must be appropriately referenced within your assignment.

5. Partial credit will be awarded for each question for which a serious attempt at finding an answer has been shown. Students are *strongly* encouraged to attempt each question and document their reasoning process even if they cannot find the correct answer. 

6. After completing the assignment, ensure that your code can run from start to finish without issue. Restart the kernal and run all cells to double check.

Name: Kathleen Ashbaker

Collaborators: Generative AI Disclaimer. This assignment was completed using the tutorial assistance of Open AI's ChatGPT Model 5.1, and supplemented with Google Search engine and Python documentation source material. All code was tested for functionality and all words in markdown cells are reflective of the author's own work, analysis and rationale. 

For this assignment, you'll need (at least) the following packages. If the package does not load, be sure it is properly installed.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

In this problem set, we will aim to build a model to explain the factors associated with a person having a heart attack.  Thereafter we also look at the predicted values and compute accuracy. We will use a dataset `heart.csv` avaialable from https://archive.ics.uci.edu/dataset/45/heart+disease, which contains health information of each person (these are the predictors) and whether or not the person had a heart attack before (the binary outcome variable). You can download the data `heart.csv` from Canvas. 

The variables are (as described on the webpage):

- age: age of the patient
- sex: sex of the patient (1 = male; 0 = female)
- cp: chest pain type chest pain type (0 = typical angina, 1 = atypical angina, 2 = non-anginal pain, 3: asymptomatic
- trestbps: resting blood pressure (in mm Hg)
- chol: cholestoral in mg/dl fetched via BMI sensor
- fbs: (fasting blood sugar > 120 mg/dl) (1 = true; 0 = false)
- restecg: resting electrocardiographic results (0 normal; 1 = having ST-T wave abnormality (T wave inversions and/or ST elevation or depression of > 0.05 mV); 2 = showing probable or definite left ventricular hypertrophy by Estes' criteria.)
- thalachh: maximum heart rate achieved
- exang: exercise induced angina (1 = yes; 0 = no)
- oldpeak: ST depression induced by exercise relative to rest
- slope:the slope of the peak exercise ST segment ( 0= upsloping; 1= flat; 2 = downsloping )
- ca: number of major vessels (range : 0-3)
- thal (Thalassemias):  0 = error (in the original dataset 0 maps to NaN's); 1 = fixed defect ; 2 = normal; 3 = reversable defect

- target: 0 = no disease, 1 = disease


### Problem 1: Logistic Regression

Points: 10

As described above, our primary aim in this assignment is to build a model to predct heart attack status -- whether a person had a heart attack or not based on other health information. 

#### (a) Load the data

Load data.  The data should contain 303 rows, and 11 columns. Do some basic checks.  Andwer these questions:
- Do we have any missing values? Are you concerns about missing data?
- What are the data types? Do these make sense for the variables?
- What are ranges of numeric variables, and possible values of categorical variables?  
- What is the rate of heart attacks among these patients?
  
Compare the values with the documentation and comment what do you see.

In [4]:
# upload the dataset 

path = "heart.csv"

df = pd.read_csv(path)

# Preview the first few rows
print(df.head())

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  


In [5]:
print(df.shape) # dimensions check of sanity 

(303, 14)


In [6]:
print(df.isna().sum()) # check for missing values 

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64


This data set has no missing values, which will make our analysis easier and more complete, thus will lead to possibly more accurate predicitons. 

In [7]:
print(df.dtypes) # look at data types in this data set 

age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca            int64
thal          int64
target        int64
dtype: object


Integers make sense for numeric types, especially discrete ones such as number of major vessels( "ca" in this data set) yet for qualititative data such as "sex" they may make less sense. Floats or decimal points make sense for data types that exist on a continuum, such as "oldpeak"

### Insert Datatype text here ! 

In [8]:
# integer and float ( numerics) variable ranges 
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns 
print(df[numeric_cols].describe().T)

          count        mean        std    min    25%    50%    75%    max
age       303.0   54.366337   9.082101   29.0   47.5   55.0   61.0   77.0
sex       303.0    0.683168   0.466011    0.0    0.0    1.0    1.0    1.0
cp        303.0    0.966997   1.032052    0.0    0.0    1.0    2.0    3.0
trestbps  303.0  131.623762  17.538143   94.0  120.0  130.0  140.0  200.0
chol      303.0  246.264026  51.830751  126.0  211.0  240.0  274.5  564.0
fbs       303.0    0.148515   0.356198    0.0    0.0    0.0    0.0    1.0
restecg   303.0    0.528053   0.525860    0.0    0.0    1.0    1.0    2.0
thalach   303.0  149.646865  22.905161   71.0  133.5  153.0  166.0  202.0
exang     303.0    0.326733   0.469794    0.0    0.0    0.0    1.0    1.0
oldpeak   303.0    1.039604   1.161075    0.0    0.0    0.8    1.6    6.2
slope     303.0    1.399340   0.616226    0.0    1.0    1.0    2.0    2.0
ca        303.0    0.729373   1.022606    0.0    0.0    0.0    1.0    4.0
thal      303.0    2.313531   0.612277

In [9]:
# possible values and counts of categorical variables ; done with the tutorial assistance of OpenAI's ChatGPT 
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'target', 'slope','ca', 'thal']

for col in cat_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())



Column: sex
sex
1    207
0     96
Name: count, dtype: int64

Column: cp
cp
0    143
2     87
1     50
3     23
Name: count, dtype: int64

Column: fbs
fbs
0    258
1     45
Name: count, dtype: int64

Column: restecg
restecg
1    152
0    147
2      4
Name: count, dtype: int64

Column: exang
exang
0    204
1     99
Name: count, dtype: int64

Column: target
target
1    165
0    138
Name: count, dtype: int64

Column: slope
slope
2    142
1    140
0     21
Name: count, dtype: int64

Column: ca
ca
0    175
1     65
2     38
3     20
4      5
Name: count, dtype: int64

Column: thal
thal
2    166
3    117
1     18
0      2
Name: count, dtype: int64


In [10]:
# Heart Attack Rate for this dataset, 'target'= 0 for no heart attack; 'target'=1 for heart attack 
if "target" in df.columns:
    print(df['target'].value_counts(normalize=True))
else:
    print("Outcome column 'target' not found.")

target
1    0.544554
0    0.455446
Name: proportion, dtype: float64


This shows that a little over 50% of participants in this data set experienced a heart attack. 

#### (b) Logistic regression model

Fit a logistic regression model with all the explanatory variables.  Do not forget to convert the categorical ones to categories!

In each case also comment on statistical significance of the results. (Hint : https://www.statsmodels.org/stable/discretemod.html )

In [11]:
import statsmodels.api as sm

# outcome
y = df['target']

# predictors: all columns except target
X = df.drop(columns=['target'])

# convert categorical predictors to dummies ; tutorial assistance here done with OpenAI's ChatGPT
X = pd.get_dummies(X, drop_first=True)

# add intercept ; tutorial assistance here done with OpenAI's ChatGPT
X = sm.add_constant(X)

# logistic regression
logit_model = sm.Logit(y, X)
result = logit_model.fit()

print(result.summary())


Optimization terminated successfully.
         Current function value: 0.348904
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:                  303
Model:                          Logit   Df Residuals:                      289
Method:                           MLE   Df Model:                           13
Date:                Tue, 18 Nov 2025   Pseudo R-squ.:                  0.4937
Time:                        20:09:48   Log-Likelihood:                -105.72
converged:                       True   LL-Null:                       -208.82
Covariance Type:            nonrobust   LLR p-value:                 7.262e-37
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          3.4505      2.571      1.342      0.180      -1.590       8.490
age           -0.0049      0.

The predictors which have statistical signigicance( p<0.05>), or probability of a heart attack are as follows: 

sex(p=0.000), cp(p=0.000), exang( p=0.017), oldpeak(p=0.012), slope(p=0.008), ca (p=0.000)

For this dataset, being male, experiencing chest pain, exercise induced angina and ST depression, having a larger number of major vessels, having thalassemia, steeper slope of peak exercise, are all strong predictors of experiencing a heart attack. 

#### (c) Data Cleaning

You probably noticed that all the above variables are coded as numbers. However, not all of these are in fact of numeric (interval, ratio) measure type. Which variables above are inherently non-numeric (nominal or ordinal)? Fix any issues.

In [12]:
# For this code block, I identified which of my explanatory variables are non-numeric; data type conversion was done using the assistance of OpenAI's Chat GPT 

# Nominal categorical variables
nominal_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal', 'target']

# Ordinal categorical variables
ordinal_cols = ['slope']

# Convert nominal to categorical
for col in nominal_cols:
    df[col] = df[col].astype('category')

# Convert ordinal with ordered categories
df['slope'] = pd.Categorical(df['slope'], 
                             categories=[0,1,2], 
                             ordered=True)

print(df.dtypes)

age            int64
sex         category
cp          category
trestbps       int64
chol           int64
fbs         category
restecg     category
thalach        int64
exang       category
oldpeak      float64
slope       category
ca          category
thal        category
target      category
dtype: object


Run EDA(Exploratory Data Analysis) to answer at least 2 questions listd below:
- How is age related to probability of heart attack?
- Are men or women more likely to have heart attack?   
- Is higher blood pressure (`trtbps`) associated with more heart attacks?
- How much more likely is heart attack for someone who has chest pain (`cp type 1`) compared to someone who has chest pain type 3?
- What is the heart attack distribution based on different (`restecg`) categories? 

For this dataset, age is not related to the probability of a heart attack( p=0.832), men are more likely to have a heart attack than women, higher blood pressure is less likely to be associated with more heart attacks ( p=0.06). 

In [13]:
import numpy as np

# This code block was partially completed using Open AI's Chat GPT strictly for tutorial's sake 

# logistic regression coefficient for cp here 
cp_coef = 0.8599  

# odds ratio for 1-unit increase in chest pain level
or_per_step = np.exp(cp_coef)

# chest pain type difference (type 3 vs type 1)
step_difference = 3 - 1

# total odds ratio
or_type3_vs_type1 = or_per_step ** step_difference

print("Odds ratio (Type 3 vs Type 1):", or_type3_vs_type1)
print(f"A person with chest pain type 3 is about {or_type3_vs_type1:.2f} times more likely to have a heart attack than someone with type 1.")


Odds ratio (Type 3 vs Type 1): 5.583411670266322
A person with chest pain type 3 is about 5.58 times more likely to have a heart attack than someone with type 1.


In [14]:
# Heart Attack Distribution of three restecg categories 

pd.crosstab(df['restecg'], df['target'], normalize='index') # cleaner, prettier version done with help from OpenAI's ChatGPT 


target,0,1
restecg,,
0,0.537415,0.462585
1,0.368421,0.631579
2,0.750000,0.250000


Having a restecg level 2( probable or definitive left ventricular hypertrohpy) is the strongest predictor of having a heart attack 


### Problem 2: Prediction

The last task is to split the dataset into a training (80% of the dataset) set and a testing (20% of the dataset) set. 

Run a logistic regression and another classification model (based on your own preference).  (Hint: Try scikit learn https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html )

Compute the confusion matrix based on the 2 models. Discuss what you find.

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)



In [16]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)


In [17]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


In [18]:
# Confusion Matrix Time!!!; tutorial provided by OpenAI ChatGPT; code further tested not long after 

from sklearn.metrics import confusion_matrix, accuracy_score

print("=== Logistic Regression Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_log))
print("Accuracy:", accuracy_score(y_test, y_pred_log))

print("\n=== Random Forest Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_rf))
print("Accuracy:", accuracy_score(y_test, y_pred_rf))


=== Logistic Regression Confusion Matrix ===
[[19  9]
 [ 3 30]]
Accuracy: 0.8032786885245902

=== Random Forest Confusion Matrix ===
[[18 10]
 [ 3 30]]
Accuracy: 0.7868852459016393


Logistic Regression had slightly higher accuracy of 80% copared to Random Forest of about 78%. The models were better at correctly predicting positive cases, yet struggled to predict negative cases, suggesting a possible class imbalance within this dataset in favor of the "target" outcome of "1" for heart attack. Logistic also captures the linear relationships between some, if not all of the predictors and experiencing a heart attack, something the random forest may not always do so, especially for small datasets. 

### Problem 3: Reflection

What concerns do you have about this data and the analysis above? How might data on heart attack status and a model like the one you fit above be used in the real world?

While I can be on board with a model like this being used in the real world to predict whether someone would experience a heart attack, one concern I have about this dataset and analysis I have is that this could be used to either overdiagnose people with signs and or symptoms of an impending heart attack, which can lead to overuse of well intentioned  healthcare interventions which may be ultimately be unneccesary ( and expensive) for the patient. This ties into the point I made earlier about a possible class imbalance. These models should be treated as guidelines and clinical decision support tools,  NOT substitutes for clinical guidance or advice. 